<a href="https://colab.research.google.com/github/contatolbonfim-ship-it/aula_git_ad/blob/main/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

In [2]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@astute-sky-513818-e3.iam.gserviceaccount.com


In [3]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "12PSvSr0k2xcObiR1mIVnhLiocCz-WiLau0CwCRrpMbo"
NOME_TABELA = "pilotos"  # Nome da primeira tabela (dimensão)

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Telemetria_Esportiva | pilotos | https://docs.google.com/spreadsheets/d/12PSvSr0k2xcObiR1mIVnhLiocCz-WiLau0CwCRrpMbo


In [6]:
# 1. Abre a planilha usando o ID correto
SHEET_ID = "12PSvSr0k2xcObiR1mIVnhLiocCz-WiLau0CwCRrpMbo"
planilha = gc.open_by_key(SHEET_ID)

# 2. Define os nomes das colunas e dados de apoio
COLUNAS_PILOTOS = ["piloto_id", "nome_ficticio", "equipe", "categoria", "pais"]
COLUNAS_VOLTAS = [
    "volta_id", "piloto_id", "corrida_id", "numero_volta",
    "tempo_volta_seg", "velocidade_media_kmh",
    "combustivel_restante_pct", "posicao_na_volta",
]

CATEGORIAS = ["kart", "formula_sim", "stock_sim", "rally_sim"]
PESO_CATEGORIAS = [30, 30, 25, 15]

PAISES = ["Brasil", "Estados Unidos", "Alemanha", "Argentina", "Itália",
          "Japão", "França", "Portugal", "México", "Espanha"]
PESO_PAISES = [35, 10, 9, 8, 8, 7, 7, 6, 5, 5]

# 3. Seleciona as abas e escreve os cabeçalhos
aba_pilotos = planilha.worksheet("pilotos")
aba_voltas = planilha.worksheet("voltas")

aba_pilotos.update(range_name="A1", values=[COLUNAS_PILOTOS])
aba_voltas.update(range_name="A1", values=[COLUNAS_VOLTAS])

print("Tudo pronto! Cabeçalhos escritos com sucesso nas abas pilotos e voltas.")

Tudo pronto! Cabeçalhos escritos com sucesso nas abas pilotos e voltas.


In [ ]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # Cria um registro fictício para as colunas definidas no esquema.
    # Personalize de acordo com o seu esquema de dados
    return {
        "coluna_1": i,
        "coluna_2": random.choice(["A", "B", "C"]),
        "coluna_3": fake.date_this_year().isoformat(),   # datas como texto
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

In [7]:
import random
from faker import Faker

# Configura os geradores aleatórios para produzir resultados reproduzíveis
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

# Listas de apoio para a telemetria esportiva
PREFIXOS = ["Fast", "Turbo", "Nitro", "Speed", "Drift", "Apex", "Boost"]
SUFIXOS = ["Pro", "Fox", "Wolf", "Ace", "Hawk", "King", "Storm"]
CATEGORIAS = ["kart", "formula_sim", "stock_sim", "rally_sim"]
PAISES = ["Brasil", "Estados Unidos", "Alemanha", "Argentina", "Itália", "Japão", "França", "Portugal"]

def gerar_linha_piloto(i):
    # Cria um registo fictício correspondente exatamente às colunas de pilotos
    return {
        "piloto_id": i,
        "nome_ficticio": f"{fake.random_element(PREFIXOS)}{fake.random_element(SUFIXOS)}{i}",
        "equipe": f"Equipe {fake.word().capitalize()} Racing",
        "categoria": random.choice(CATEGORIAS),
        "pais": random.choice(PAISES),
    }

# Gera 10 registos numerados de 1 a 10
linhas_pilotos = [gerar_linha_piloto(i) for i in range(1, 11)]

# Organiza os valores na ordem exata do cabeçalho da tabela pilotos
valores_pilotos = [[linha[c] for c in COLUNAS_PILOTOS] for linha in linhas_pilotos]

# Acrescenta os registos à aba de pilotos na planilha
aba_pilotos.append_rows(valores_pilotos, value_input_option="RAW")
print(len(valores_pilotos), "linhas de teste enviadas para a aba pilotos!")

10 linhas de teste enviadas para a aba pilotos!


In [8]:
from google.colab import files
import pandas as pd

print("Por favor, selecione APENAS o arquivo CSV de PILOTOS:")
uploaded_pilotos = files.upload()
nome_arq_pilotos = list(uploaded_pilotos.keys())[0]

# Lê e envia para a aba pilotos
df_pilotos = pd.read_csv(nome_arq_pilotos)
aba_pilotos.clear()
aba_pilotos.update(range_name="A1", values=[list(df_pilotos.columns)])
aba_pilotos.append_rows(df_pilotos.astype(object).values.tolist(), value_input_option="RAW")

print(f"Sucesso! {len(df_pilotos)} linhas enviadas para a aba 'pilotos'.")

Por favor, selecione APENAS o arquivo CSV de PILOTOS:


Saving pilotos.csv to pilotos.csv
Sucesso! 10800 linhas enviadas para a aba 'pilotos'.


In [9]:
import time
from gspread.exceptions import APIError

print("Por favor, selecione APENAS o arquivo CSV de VOLTAS:")
uploaded_voltas = files.upload()
nome_arq_voltas = list(uploaded_voltas.keys())[0]

# Lê o arquivo de voltas
df_voltas = pd.read_csv(nome_arq_voltas)
aba_voltas.clear()
aba_voltas.update(range_name="A1", values=[list(df_voltas.columns)])

# Envia em lotes seguros para o Google Sheets (devido ao tamanho grande)
linhas_voltas = df_voltas.astype(object).values.tolist()
total = len(linhas_voltas)
tamanho_lote = 5000

for inicio in range(0, total, tamanho_lote):
    lote = linhas_voltas[inicio:inicio + tamanho_lote]
    for tentativa in range(5):
        try:
            aba_voltas.append_rows(lote, value_input_option="RAW")
            break
        except APIError as erro:
            print("Limite do Google atingido. Esperando 60s...", erro)
            time.sleep(60)
    print(f"[voltas] {min(inicio + tamanho_lote, total)}/{total} linhas enviadas")
    time.sleep(2)

print(f"Sucesso! Todas as {total} linhas enviadas para a aba 'voltas'.")

Por favor, selecione APENAS o arquivo CSV de VOLTAS:


Saving voltas.csv to voltas.csv
[voltas] 5000/55000 linhas enviadas
[voltas] 10000/55000 linhas enviadas
[voltas] 15000/55000 linhas enviadas
[voltas] 20000/55000 linhas enviadas
[voltas] 25000/55000 linhas enviadas
[voltas] 30000/55000 linhas enviadas
[voltas] 35000/55000 linhas enviadas
[voltas] 40000/55000 linhas enviadas
[voltas] 45000/55000 linhas enviadas
[voltas] 50000/55000 linhas enviadas
[voltas] 55000/55000 linhas enviadas
Sucesso! Todas as 55000 linhas enviadas para a aba 'voltas'.
